##  Étape 1 — Configuration du notebook3.1 Préparation des données



In [ ]:
# 3. Analyse de la qualité et exploration des données

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "README.md").exists():
    ROOT = ROOT.parent
FIG_ROOT = ROOT / "analyse_exploratoire" / "figures"
DATA_EXPORT_ROOT = ROOT / "data" / "processed" / "analyse_exploratoire"
FIG_ROOT.mkdir(parents=True, exist_ok=True)
DATA_EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

import numpy as np
from zoneinfo import ZoneInfo



def load_all(DATA_DIR):
    """
    Charge tous les fichiers CSV ou Excel d'un dossier et les concatène.
    Ajoute la colonne 'fichier_originaire' pour tracer l’origine des données.
    """
    paths = sorted([p for p in Path(DATA_DIR).rglob("*") if p.suffix.lower() in (".csv", ".xlsx", ".xls")])
    dfs = []

    for p in paths:
        try:
            if p.suffix.lower() in (".xlsx", ".xls"):
                df = pd.read_excel(p)
            else:
                df = pd.read_csv(p, sep=None, engine="python")
            df["fichier_originaire"] = p.name
            dfs.append(df)
        except Exception as e:
            print(f" Erreur lors de la lecture de {p.name} : {e}")

    if not dfs:
        raise ValueError(f"Aucun fichier valide trouvé dans {DATA_DIR}")

    return pd.concat(dfs, ignore_index=True)


def assign_mslot_from_filename_winter(file_name: str) -> str:
    """
    Détermine le créneau horaire M1..M4 à partir du nom du fichier.
    Format attendu : picopatt_montpellier_<parcours>_YYYYMMDD_HHMM.csv
    (valable pour les mesures d'hiver uniquement)
    """
    m = re.search(r"(\d{8})_(\d{4})", file_name)
    if not m:
        return "UNK"

    _, time_part = m.groups()
    hour = int(time_part[:2])

    if 8 <= hour < 11:
        return "M1"
    elif 11 <= hour < 14:
        return "M2"
    elif 14 <= hour < 17:
        return "M3"
    elif 17 <= hour < 20:
        return "M4"
    else:
        return "UNK"

## Étape 2 — Chargement des données nettoyées

In [ ]:
#  Dossier contenant les fichiers nettoyés sans zéros
DATA_NOZERO = ROOT / "data" / "processed" / "picopatt" / "clean_nozeros"

# Vérifie que le dossier existe et contient des fichiers
if not DATA_NOZERO.exists():
    raise FileNotFoundError(f" Le dossier {DATA_NOZERO} n'existe pas.")
if not any(DATA_NOZERO.glob("*.csv")):
    raise FileNotFoundError(f"Aucun fichier .csv trouvé dans {DATA_NOZERO}")

print(f" Chargement des fichiers depuis : {DATA_NOZERO}")

# Chargement de tous les fichiers
raw = load_all(DATA_NOZERO)

# Conversion en datetime (si colonne "timestamp" existe)
if "timestamp" in raw.columns:
    raw["timestamp"] = pd.to_datetime(raw["timestamp"], errors="coerce", dayfirst=True)
else:
    print("Aucune colonne 'timestamp' trouvée dans les fichiers.")

# Création des colonnes utiles à partir du nom du fichier
raw["M_slot"] = raw["fichier_originaire"].apply(assign_mslot_from_filename_winter)
raw["date"] = raw["timestamp"].dt.tz_localize(None).dt.date  # supprime timezone si existante

# Résumé global
print("\n Données chargées :", raw.shape)
print(" Période :", raw["timestamp"].min(), "→", raw["timestamp"].max())
if "track_id" in raw.columns:
    print(" Parcours disponibles :", raw["track_id"].dropna().unique())
else:
    print("Aucune colonne 'track_id' détectée.")

## Étape 3.1 — Vérification des valeurs manquantes et des zéros

In [ ]:
# Variables météo à surveiller
METEO_VARS = [
    "tair_thermohygro", "tair_tc1", "tair_tc2", "tair_anemo",
    "rh_thermohygro", "ws", "wdir",
    "sw_up", "sw_front", "sw_right",
    "lw_up", "lw_down", "lw_front", "lw_back", "lw_left", "lw_right",
    "tmrt", "pet"
]

# Ne garder que celles réellement présentes dans le DataFrame
METEO_VARS = [v for v in METEO_VARS if v in raw.columns]

print(f" Variables météo analysées ({len(METEO_VARS)} présentes) : {METEO_VARS}")

# Pourcentage de valeurs manquantes 
missing_pct = raw[METEO_VARS].isna().mean().sort_values(ascending=False) * 100
print("\nPourcentage de valeurs manquantes (Top 10) :")
display(missing_pct.head(10))

# Pourcentage de valeurs nulles (égales à zéro) 
zero_pct = (raw[METEO_VARS] == 0).mean().sort_values(ascending=False) * 100
print("\nPourcentage de valeurs égales à 0 (Top 10) :")
display(zero_pct.head(10))

# Fichiers contenant encore des 0 
zero_files = (
    raw.loc[raw[METEO_VARS].eq(0).any(axis=1), "fichier_originaire"]
       .value_counts()
       .head(10)
)
print("\nFichiers contenant encore des zéros (Top 10) :")
display(zero_files)

# Fichiers contenant des valeurs manquantes
nan_files = (
    raw.loc[raw[METEO_VARS].isna().any(axis=1), "fichier_originaire"]
       .value_counts()
       .head(10)
)
print("\nFichiers contenant des valeurs manquantes (Top 10) :")

## Étape 3.2 — Vérification de la cohérence temporelle

In [ ]:
#  Création du dossier de sortie
OUTPUT_DIR = FIG_ROOT / "comptage"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#  Vérification des colonnes nécessaires
required_cols = {"fichier_originaire", "track_id", "date", "M_slot"}
missing_cols = required_cols - set(raw.columns)

# Si certaines colonnes manquent, on les reconstruit depuis le nom du fichier
if missing_cols:
    print(f" Colonnes manquantes détectées : {missing_cols}")
    if "fichier_originaire" not in raw.columns:
        raise ValueError("La colonne 'fichier_originaire' est nécessaire pour extraire les infos manquantes.")
    
    def parse_filename(name):
        """Extrait track_id, date et M_slot depuis le nom du fichier."""
        m = re.search(r"montpellier_(\w+)_(\d{8})_(\d{4})", name)
        if not m:
            return pd.Series([None, None, None])
        track, date_str, time_str = m.groups()
        date = pd.to_datetime(date_str, format="%Y%m%d").date()
        hour = int(time_str[:2])
        if 8 <= hour < 11:
            mslot = "M1"
        elif 11 <= hour < 14:
            mslot = "M2"
        elif 14 <= hour < 17:
            mslot = "M3"
        elif 17 <= hour < 20:
            mslot = "M4"
        else:
            mslot = "UNK"
        return pd.Series([track, date, mslot])

    raw[["track_id", "date", "M_slot"]] = raw.apply(
        lambda row: parse_filename(row["fichier_originaire"]), axis=1
    )

# Force le bon format de date
raw["date"] = pd.to_datetime(raw["date"], errors="coerce").dt.date

# Nombre de créneaux M1–M4 distincts par jour et par parcours
nb_passages = (
    raw.dropna(subset=["track_id", "date", "M_slot"])
       .groupby(["track_id", "date"])
       .agg(nb_M=("M_slot", "nunique"))
       .reset_index()
)

print("Nombre de créneaux M1–M4 par jour et par parcours :")
display(nb_passages.head(10))

# Résumé global
nb_passages_summary = (
    nb_passages.groupby("track_id")["nb_M"]
               .value_counts()
               .unstack(fill_value=0)
               .sort_index(axis=1)
)

print("\nRépartition du nombre de passages par jour :")
display(nb_passages_summary)

#  Heatmap binaire de présence par jour et M_slot
for track in sorted(raw["track_id"].dropna().unique()):
    df_t = (
        raw.query("track_id == @track")
           .dropna(subset=["date", "M_slot"])
           .groupby(["date", "M_slot"])
           .size()
           .unstack(fill_value=0)
           .reindex(columns=["M1", "M2", "M3", "M4"])
    )

    # Heatmap 
    plt.figure(figsize=(6, max(3, len(df_t) * 0.3)))
    sns.heatmap(
        (df_t > 0).astype(int),             
        cmap=["#FFFFCC", "#00194A"],       
        cbar=False,
        linewidths=0.2,
        linecolor="white",
        vmin=0, vmax=1
    )
    plt.title(f"Présence des passages par jour — {track}")
    plt.xlabel("Créneau horaire (M_slot)")
    plt.ylabel("Date")
    plt.tight_layout()

    # Sauvegarde dans le dossier comptage 
    out_path = OUTPUT_DIR / f"heatmap_passages_{track}.png"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f" Heatmap enregistrée : {out_path.name}")

    plt.show()

## l’étape 3.3 — Agrégation par passage.

In [ ]:
# Répertoire contenant les fichiers à agréger
DATA_DIR = ROOT / "data" / "processed" / "picopatt" / "clean_nozeros"

# Liste des fichiers CSV à inclure (sauf celui à exclure)
paths = [
    p for p in DATA_DIR.glob("*.csv")
    if "picopatt_montpellier_ecusson_20250114_0832_clean.csv" not in p.name
]

print(f"Fichiers pris en compte ({len(paths)}):")
for p in paths:
    print("  -", p.name)

# Lecture et concaténation des fichiers
frames = []
for p in paths:
    df = pd.read_csv(p)
    df["source_file"] = p.name
    frames.append(df)

if not frames:
    raise ValueError("Aucun fichier n’a été trouvé pour l’agrégation.")

raw = pd.concat(frames, ignore_index=True, sort=False)
print(f"\nDonnées combinées : {raw.shape[0]} lignes, {raw.shape[1]} colonnes.")

# Définition des variables météo
METEO_VARS = [
    "tair_thermohygro", "tair_tc1", "tair_tc2", "tair_anemo",
    "rh_thermohygro", "ws", "wdir",
    "sw_up", "sw_front", "sw_right",
    "lw_up", "lw_down", "lw_front", "lw_back", "lw_left", "lw_right",
    "tmrt", "pet"
]

# Vérifie que les colonnes existent bien
METEO_VARS = [v for v in METEO_VARS if v in raw.columns]
if not METEO_VARS:
    raise ValueError("Aucune variable météo trouvée dans les fichiers.")

print(f"\nVariables météo utilisées ({len(METEO_VARS)}) : {METEO_VARS}")

# Agrégation par parcours, date et créneau horaire 
agg_passage = (
    raw.dropna(subset=["track_id", "date", "M_slot"])
       .groupby(["track_id", "date", "M_slot"])[METEO_VARS]
       .agg(["mean", "std", "min", "max"])
)

# Renommage clair des colonnes
agg_passage.columns = [f"{col}_{stat}" for col, stat in agg_passage.columns]
agg_passage = agg_passage.reset_index()

# Sauvegarde
AGG_DIR = DATA_EXPORT_ROOT / "aggregations"
AGG_DIR.mkdir(parents=True, exist_ok=True)

output_file = AGG_DIR / "agg_par_passage.csv"
agg_passage.to_csv(output_file, index=False)

# Résumé
print(f"\nFichier sauvegardé : {output_file}")
print(f"Taille du tableau agrégé : {agg_passage.shape}")
print("\nExtrait des 5 premières lignes :")
display(agg_passage.head())

print("\nNombre de combinaisons uniques (track_id, date, M_slot) :",
      agg_passage[['track_id', 'date', 'M_slot']].drop_duplicates().shape[0])

## l’étape 3.3 — Agrégation par passage (fichier exclu)

In [ ]:
# Dossier des fichiers propres
DATA_DIR = ROOT / "data" / "processed" / "picopatt" / "clean_nozeros"

# Liste des fichiers à inclure (exclut le fichier fautif) 
paths = [
    p for p in DATA_DIR.glob("*.csv")
    if "picopatt_montpellier_ecusson_20250114_0832.csv" not in p.name
]

print(f"{len(paths)} fichiers détectés (le fichier fautif est exclu).")
for p in paths:
    print("  -", p.name)

# Lecture et concaténation
frames = []
for p in paths:
    try:
        df = pd.read_csv(p, sep=None, engine="python")
        df["fichier_originaire"] = p.name
        frames.append(df)
    except Exception as e:
        print(f" Erreur lecture {p.name}: {e}")

if not frames:
    raise ValueError("Aucun fichier valide trouvé dans le dossier !")

raw_filtered = pd.concat(frames, ignore_index=True, sort=False)
print(f"\nDonnées combinées : {raw_filtered.shape[0]} lignes, {raw_filtered.shape[1]} colonnes.\n")

# Définition des variables météo 
METEO_VARS = [
    "tair_thermohygro", "tair_tc1", "tair_tc2", "tair_anemo",
    "rh_thermohygro", "ws", "wdir",
    "sw_up", "sw_front", "sw_right",
    "lw_up", "lw_down", "lw_front", "lw_back", "lw_left", "lw_right",
    "tmrt", "pet"
]
METEO_VARS = [v for v in METEO_VARS if v in raw_filtered.columns]
print(f" Variables utilisées ({len(METEO_VARS)}): {METEO_VARS}")

# Agrégation
agg_passage_clean = (
    raw_filtered.dropna(subset=["track_id", "date", "M_slot"])
                .groupby(["track_id", "date", "M_slot"])[METEO_VARS]
                .agg(["mean", "std", "min", "max"])
)
agg_passage_clean.columns = [f"{col}_{stat}" for col, stat in agg_passage_clean.columns]
agg_passage_clean = agg_passage_clean.reset_index()

# Sauvegarde
AGG_DIR = DATA_EXPORT_ROOT / "aggregations"
AGG_DIR.mkdir(parents=True, exist_ok=True)

output_file = AGG_DIR / "agg_par_passage_clean_no_ecusson0832.csv"
agg_passage_clean.to_csv(output_file, index=False)

print(f"Fichier agrégé créé : {output_file}")
print(f"Taille du tableau : {agg_passage_clean.shape}")

## M_slot valide 

In [ ]:
# Vérifie que la colonne existe
if "M_slot" not in raw.columns:
    raise ValueError(" La colonne 'M_slot' est absente du DataFrame. Vérifie ton chargement de données.")

# Détection des valeurs manquantes ou inconnues
mask_missing = raw["M_slot"].isna() | (raw["M_slot"] == "UNK")
total_missing = mask_missing.sum()
print(f"Total de lignes sans M_slot valide : {total_missing:,}")

if total_missing > 0:
    # Regroupement par fichier pour identifier les sources problématiques
    missing_by_file = (
        raw.loc[mask_missing]
           .groupby("fichier_originaire", dropna=False)
           .size()
           .sort_values(ascending=False)
           .rename("nb_lignes_sans_Mslot")
           .reset_index()
    )

    print("\n Fichiers contenant des lignes sans M_slot ou 'UNK' :")
    display(missing_by_file.head(20))

    # Correction automatique à partir du nom du fichier
    def infer_mslot_from_filename(filename):
        """Déduit M_slot selon l'heure contenue dans le nom du fichier."""
        m = re.search(r"_(\d{8})_(\d{4})", str(filename))
        if not m:
            return None
        _, time_str = m.groups()
        hour = int(time_str[:2])
        if 8 <= hour < 11:
            return "M1"
        elif 11 <= hour < 14:
            return "M2"
        elif 14 <= hour < 17:
            return "M3"
        elif 17 <= hour < 20:
            return "M4"
        else:
            return None

    # Correction sur les lignes concernées
    raw.loc[mask_missing, "M_slot"] = raw.loc[mask_missing, "fichier_originaire"].apply(infer_mslot_from_filename)

    # Vérifie le résultat après correction
    mask_missing_after = raw["M_slot"].isna() | (raw["M_slot"] == "UNK")
    remaining = mask_missing_after.sum()
    print(f"\n Correction effectuée. Lignes restantes sans M_slot valide : {remaining:,}")

    # Détail si certaines lignes n'ont toujours pas pu être corrigées
    if remaining > 0:
        print("\nLignes non corrigées (fichiers non reconnus) :")
        display(raw.loc[mask_missing_after, ["fichier_originaire", "M_slot"]].drop_duplicates())
else:
    print(" Aucun M_slot manquant ni 'UNK' détecté !")
    #  Affiche un échantillon des lignes qui ont été corrigées

    
mask_corr = raw["fichier_originaire"].str.contains("boulevards_20241114_1728", na=False)
display(raw.loc[mask_corr, ["fichier_originaire", "timestamp", "M_slot"]].head(10))

##  Vérification finale des 0 et NaN dans clean_nozeros

In [ ]:

DATA_NOZERO = ROOT / "data" / "processed" / "picopatt" / "clean_nozeros"
REPORT_DIR = DATA_EXPORT_ROOT / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Lecture sûre des fichiers 
def read_any(p: Path) -> pd.DataFrame:
    """Lit un fichier CSV ou Excel, en détectant automatiquement le séparateur."""
    try:
        if p.suffix.lower() in (".xlsx", ".xls"):
            df = pd.read_excel(p)
        else:
            df = pd.read_csv(p, sep=None, engine="python")
        df["__source_file"] = p.name
        return df
    except Exception as e:
        print(f"Erreur lecture {p.name}: {e}")
        return pd.DataFrame()

# Fusion de tous les fichiers
def load_all(data_dir: Path) -> pd.DataFrame:
    """Charge et fusionne tous les fichiers du dossier."""
    paths = sorted([p for p in data_dir.rglob("*") if p.suffix.lower() in (".csv", ".xlsx", ".xls")])
    if not paths:
        raise FileNotFoundError(f"Aucun fichier trouvé dans {data_dir.resolve()}")

    frames = [read_any(p) for p in paths if not read_any(p).empty]
    if not frames:
        raise ValueError("Aucun fichier lisible trouvé dans le dossier.")
    return pd.concat(frames, ignore_index=True, sort=False)

# Chargement
raw_nozero = load_all(DATA_NOZERO)
print(f"Données chargées : {len(raw_nozero):,} lignes, {len(raw_nozero.columns)} colonnes")

# Sélection des colonnes numériques
numeric_cols = raw_nozero.select_dtypes(include=[np.number]).columns
if numeric_cols.empty:
    raise ValueError("Aucune colonne numérique trouvée dans les fichiers !")

# Comptage des 0 et des NaN
zero_counts = (raw_nozero[numeric_cols] == 0).sum()
nan_counts = raw_nozero[numeric_cols].isna().sum()

# Calcul des pourcentages
total = len(raw_nozero)
zero_pct = (zero_counts / total * 100).round(2)
nan_pct = (nan_counts / total * 100).round(2)

# Assemblage du rapport
report = (
    pd.DataFrame({
        "nb_zeros": zero_counts,
        "pct_zeros": zero_pct,
        "nb_nans": nan_counts,
        "pct_nans": nan_pct
    })
    .query("nb_zeros > 0 or nb_nans > 0")
    .sort_values(by=["pct_zeros", "pct_nans"], ascending=False)
)

# Affichage
if report.empty:
    print(" Aucune variable ne contient de 0 ni de NaN !")
else:
    print("\nVariables contenant encore des 0 ou des NaN :")
    display(report.head(30))
    print(f"\n{report.shape[0]} variables concernées sur {len(numeric_cols)} colonnes numériques.")

# Sauvegarde du rapport
report_path = REPORT_DIR / "zero_nan_report.csv"
report.to_csv(report_path, index=True)
print(f"\nRapport sauvegardé dans : {report_path}")
print(f"Taille totale du dataset : {total:,} lignes")

## lecture du fichier 

In [ ]:
# Lecture du fichier agrégé
agg_file = DATA_EXPORT_ROOT / "aggregations" / "agg_par_passage.csv"
agg = pd.read_csv(agg_file)

print(f" Données chargées : {len(agg):,} lignes, {len(agg.columns)} colonnes")

# Liste des variables météo (moyennes uniquement)
METEO_VARS = [
    "tair_thermohygro_mean", "tair_tc1_mean", "tair_tc2_mean", "tair_anemo_mean",
    "rh_thermohygro_mean", "ws_mean", "wdir_mean",
    "sw_up_mean", "sw_front_mean", "sw_right_mean",
    "lw_up_mean", "lw_down_mean", "lw_front_mean", "lw_back_mean", "lw_left_mean", "lw_right_mean",
    "tmrt_mean", "pet_mean"
]
METEO_VARS = [v for v in METEO_VARS if v in agg.columns]
print(f"Variables trouvées : {METEO_VARS}")

##  Boxplot par capteur — détection visuelle d’anomalies

Ce graphique met en évidence les valeurs extrêmes (comme 533 °C et 112 °C) directement.
C’est la meilleure manière de montrer “attention, ces points sortent complètement du lot”.

In [ ]:
#  Charger le fichier d’agrégation complet 
agg_path = DATA_EXPORT_ROOT / "aggregations" / "agg_par_passage.csv"
agg = pd.read_csv(agg_path)

#  Variables critiques (les capteurs à comparer)
vars_to_plot = ["tair_thermohygro_mean", "tair_tc1_mean", "tair_tc2_mean"]

# Vérification que les colonnes existent dans le fichier
vars_to_plot = [v for v in vars_to_plot if v in agg.columns]
if not vars_to_plot:
    raise ValueError("Aucune des variables demandées n'existe dans le fichier agrégé.")

# Mise en forme 
df_melt = agg.melt(
    id_vars=["track_id"],
    value_vars=vars_to_plot,
    var_name="Capteur",
    value_name="Température"
)

# Graphique boxplot 
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df_melt,
    x="Capteur",
    y="Température",
    hue="track_id",
    palette="Set2",
    showfliers=True  # on garde les points aberrants visibles
)
plt.title("Détection visuelle des valeurs aberrantes — Températures moyennes")
plt.ylabel("Température (°C)")
plt.xlabel("")
plt.legend(title="Secteur", loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Charger le fichier d’agrégation
agg_path = DATA_EXPORT_ROOT / "aggregations" / "agg_par_passage.csv"
agg = pd.read_csv(agg_path)

# Variables "_mean" uniquement
mean_vars = [col for col in agg.columns if "_mean" in col]

if not mean_vars:
    raise ValueError("Aucune variable contenant '_mean' n’a été trouvée dans le fichier.")

# Fonction de détection des outliers (IQR)
def detect_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[column] < lower) | (df[column] > upper)]
    return outliers, lower, upper

# Boucle sur chaque variable "_mean"
for var in mean_vars:
    outliers, lower, upper = detect_outliers_iqr(agg, var)
    n_outliers = len(outliers)

    # Graphique boxplot
    plt.figure(figsize=(6, 4))
    sns.boxplot(
        data=agg,
        x="track_id",
        y=var,
        hue="track_id",     
        legend=False,         
        palette="Set2",
        showfliers=True
    )
    plt.title(f"{var}")
    plt.ylabel("Mean value")
    plt.xlabel("Track ID")
    plt.tight_layout()
    plt.show()

##  Boxplot par capteur — détection visuelle apres 

In [ ]:
# Charger ton fichier d'agrégation complet 
agg_path = DATA_EXPORT_ROOT / "aggregations" / "agg_par_passage_clean_no_ecusson0832.csv"
agg = pd.read_csv(agg_path)

# Exclure les données issues du fichier fautif (si la colonne existe)
if "fichier_originaire" in agg.columns:
    agg = agg[~agg["fichier_originaire"].str.contains("picopatt_montpellier_ecusson_20250114_0832", case=False, na=False)]
    print("Données du fichier fautif supprimées avant visualisation.")
else:
    print(" Colonne 'fichier_originaire' absente — on suppose que les données agrégées ne contiennent plus ce fichier.")

# Variables critiques (les capteurs à comparer)
vars_to_plot = ["tair_thermohygro_mean", "tair_tc1_mean", "tair_tc2_mean"]

# Vérification de la présence des colonnes
vars_to_plot = [v for v in vars_to_plot if v in agg.columns]
if not vars_to_plot:
    raise ValueError("Aucune variable valide trouvée dans le fichier d'agrégation.")

# Mise en forme (long format)
df_melt = agg.melt(
    id_vars=["track_id"],
    value_vars=vars_to_plot,
    var_name="Capteur",
    value_name="Température"
)

# Graphique boxplot
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df_melt,
    x="Capteur",
    y="Température",
    hue="track_id",
    palette="Set2",
    showfliers=True
)
plt.title("Détection visuelle des valeurs aberrantes — Températures moyennes (fichier fautif exclu)")
plt.ylabel("Température (°C)")
plt.xlabel("")
plt.legend(title="Secteur", loc="upper right")
plt.tight_layout()
plt.show()

## Scatterplot — Relation Tmrt / PET

In [ ]:

plt.figure(figsize=(12,6))
sns.scatterplot(data=agg, x="tmrt_mean", y="pet_mean", hue="track_id", alpha=0.8, palette="Set2")

# Ligne de référence
plt.axline((0,0), slope=1, color="gray", linestyle="--", lw=1, label="y = x (corrélation idéale)")

# Annotation de la valeur extrême
plt.scatter(13.7, -0.41, color="red", s=80, zorder=5, label="Valeur  PET = -0.41 °C (Écusson, 14/01/2025)")

plt.title("Lien entre la température radiante moyenne (Tmrt) et le PET")
plt.xlabel("Tmrt (°C)")
plt.ylabel("PET (°C)")
plt.legend()
plt.tight_layout()
plt.show()

##   Évolution temporelle par variable — Un seul créneau horaire (M1, M2, M3 ou M4)

In [ ]:


#  Charger le fichier agrégé 
agg_file = DATA_EXPORT_ROOT / "aggregations" / "agg_par_passage.csv"
agg = pd.read_csv(agg_file)

# Convertir la colonne date
agg["date"] = pd.to_datetime(agg["date"], errors="coerce")

# Choisir le créneau horaire à analyser
CRENEAU = "M1" 

# Filtrer uniquement ce créneau
subset = agg.query("M_slot == @CRENEAU")

# Variables à tracer 
vars_to_plot = [
    "tair_thermohygro_mean", "tair_tc1_mean", "tair_tc2_mean", "tair_anemo_mean",
    "rh_thermohygro_mean", "ws_mean", "wdir_mean",
    "sw_up_mean", "sw_front_mean", "sw_right_mean",
    "lw_up_mean", "lw_down_mean", "lw_front_mean", "lw_back_mean", "lw_left_mean", "lw_right_mean",
    "tmrt_mean", "pet_mean"
]

#  Vérifier que les colonnes existent
vars_to_plot = [v for v in vars_to_plot if v in subset.columns]
print(f"Variables disponibles ({len(vars_to_plot)}): {vars_to_plot}")

# Style graphique
sns.set(style="whitegrid", font_scale=1.1)

# Tracé pour chaque variable
for var in vars_to_plot:
    plt.figure(figsize=(10,6))
    sns.lineplot(
        data=subset,
        x="date",
        y=var,
        hue="track_id",
        marker="o",
        palette="Set2"
    )
    plt.title(f"Évolution temporelle de {var} — Créneau {CRENEAU}")
    plt.xlabel("Date de mesure")
    plt.ylabel(var.replace("_mean", " (moyenne)"))
    plt.legend(title="Parcours", bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Évolution temporelle — Créneau M2

# Charger le fichier agrégé
agg_file = DATA_EXPORT_ROOT / "aggregations" / "agg_par_passage.csv"
agg = pd.read_csv(agg_file)
agg["date"] = pd.to_datetime(agg["date"], errors="coerce")

# Filtrer sur le créneau M2
subset = agg[agg["M_slot"] == "M2"]

# Variables à tracer
vars_to_plot = [
    "tair_thermohygro_mean", "tair_tc1_mean", "tair_tc2_mean", "tair_anemo_mean",
    "rh_thermohygro_mean", "ws_mean", "wdir_mean",
    "sw_up_mean", "sw_front_mean", "sw_right_mean",
    "lw_up_mean", "lw_down_mean", "lw_front_mean", "lw_back_mean", "lw_left_mean", "lw_right_mean",
    "tmrt_mean", "pet_mean"
]

vars_to_plot = [v for v in vars_to_plot if v in subset.columns]
sns.set(style="whitegrid", font_scale=1.1)

for var in vars_to_plot:
    plt.figure(figsize=(10,6))
    sns.lineplot(
        data=subset,
        x="date",
        y=var,
        hue="track_id",
        marker="o",
        palette="Set2"
    )
    plt.title(f"Évolution temporelle de {var} — Créneau M2")
    plt.xlabel("Date de mesure")
    plt.ylabel(var.replace("_mean", " (moyenne)"))
    plt.legend(title="Parcours", bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Évolution temporelle — Créneau M3


subset = agg[agg["M_slot"] == "M3"]
vars_to_plot = [v for v in vars_to_plot if v in subset.columns]

for var in vars_to_plot:
    plt.figure(figsize=(10,6))
    sns.lineplot(
        data=subset,
        x="date",
        y=var,
        hue="track_id",
        marker="o",
        palette="Set2"
    )
    plt.title(f"Évolution temporelle de {var} — Créneau M3")
    plt.xlabel("Date de mesure")
    plt.ylabel(var.replace("_mean", " (moyenne)"))
    plt.legend(title="Parcours", bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Évolution temporelle — Créneau M4

subset = agg[agg["M_slot"] == "M4"]
vars_to_plot = [v for v in vars_to_plot if v in subset.columns]

for var in vars_to_plot:
    plt.figure(figsize=(10,6))
    sns.lineplot(
        data=subset,
        x="date",
        y=var,
        hue="track_id",
        marker="o",
        palette="Set2"
    )
    plt.title(f"Évolution temporelle de {var} — Créneau M4")
    plt.xlabel("Date de mesure")
    plt.ylabel(var.replace("_mean", " (moyenne)"))
    plt.legend(title="Parcours", bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# fichier source
f = ROOT / "data" / "processed" / "picopatt" / "clean_nozeros" / "picopatt_montpellier_ecusson_20250114_0832.csv"
df = pd.read_csv(f)

print(df.columns)

## Correction de l' aggregation

In [ ]:
# Charger le fichier original 
f = ROOT / "data" / "processed" / "picopatt" / "clean_nozeros" / "picopatt_montpellier_ecusson_20250114_0832.csv"
df = pd.read_csv(f)

# Identifier les colonnes concernées
cols_to_clean = [c for c in df.columns if any(x in c.lower() for x in ["tair_tc1", "tair_tc2"])]

print(f"Colonnes détectées pour correction : {cols_to_clean}")

# Vérification avant correction
for col in cols_to_clean:
    print(f"\nAvant correction : {col} ")
    print(df[col].describe())

#  Correction : mise à NaN de toutes les valeurs suspectes
for col in cols_to_clean:
    df[col] = np.nan
    print(f" {col} : toutes les valeurs remplacées par NaN")

#  Création d’un nouveau fichier nettoyé 
f_clean = f.with_name(f"{f.stem}_clean{f.suffix}")
df.to_csv(f_clean, index=False)

print(f"\n Nouveau fichier créé : {f_clean}")

#  Vérification rapide après correction 
for col in cols_to_clean:
    print(f"\n Après correction : {col} ")
    print(df[col].describe())

## Localisation ilots de fraicheur

In [ ]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

# Dossier contenant tes fichiers
DATA_DIR = ROOT / "data" / "processed" / "picopatt" / "clean_nozeros"

# Paramètres d’analyse
secteur = "antigone"
sections_cibles = [3, 27, 33]
passages = ["M1", "M2", "M3", "M4"]

# Colonnes de température
temp_cols = ["tair_thermohygro", "tair_tc1", "tair_tc2", "tair_anemo"]

# Liste des fichiers du secteur
csv_files = sorted([f for f in DATA_DIR.glob("*.csv") if secteur in f.name.lower()])

print(f"{len(csv_files)} fichiers '{secteur.capitalize()}' détectés.\n")

# Conteneur global
all_data = []

# Lecture de tous les fichiers
for file in csv_files:
    try:
        df = pd.read_csv(file, sep=None, engine="python")
        df["fichier_originaire"] = file.name

        # Vérification des colonnes essentielles
        required_cols = ["section_id", "track_id", "M_slot", "lon_ontrack", "lat_ontrack"]
        if not all(col in df.columns for col in required_cols):
            continue

        # Filtrer uniquement le parcours et les sections ciblées
        df = df[(df["track_id"].str.lower() == secteur) & (df["section_id"].isin(sections_cibles))]

        if df.empty:
            continue

        # Moyenne de température par point
        available_temps = [t for t in temp_cols if t in df.columns]
        if not available_temps:
            continue

        df["t_moy"] = df[available_temps].mean(axis=1)

        # Garder les colonnes utiles
        df = df[["section_id", "M_slot", "t_moy", "lon_ontrack", "lat_ontrack", "fichier_originaire"]]
        all_data.append(df)

    except Exception as e:
        print(f"Erreur dans {file.name} : {e}")

# Agrégation et visualisation par passage
if all_data:
    full_df = pd.concat(all_data, ignore_index=True)

    for passage in passages:
        sub_df = full_df[full_df["M_slot"] == passage]

        if sub_df.empty:
            print(f"Aucun point pour {passage}")
            continue

        # Calcul des statistiques par section
        summary = (
            sub_df.groupby("section_id", observed=True)
                  .agg(t_moyenne=("t_moy", "mean"),
                       t_min=("t_moy", "min"),
                       t_max=("t_moy", "max"),
                       n_points=("t_moy", "count"))
                  .reset_index()
                  .sort_values("section_id")
        )

        print(f"\n Températures moyennes - Antigone / Passage {passage} :\n")
        print(summary.to_string(index=False))

        # --- Graphique ---
        plt.figure(figsize=(6, 4))
        plt.bar(
            summary["section_id"].astype(str),
            summary["t_moyenne"],
            color=["royalblue" if s in [3, 27] else "orange" for s in summary["section_id"]],
            alpha=0.8
        )
        plt.title(f"Températures moyennes - Passage {passage} (Antigone)")
        plt.ylabel("Température moyenne (°C)")
        plt.xlabel("Sections")
        plt.grid(axis="y", linestyle="--", alpha=0.4)
        plt.tight_layout()
        plt.show()

else:
    print(" Aucune donnée trouvée pour ces sections.")